# 04 • Criar colunas e tratar textos

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [1]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [2]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
Dimensões: (344, 8)
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Series
   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen
                   ausentes  percentual
species                   0    0.000000
island                    0    0.000000
bill_length_mm            2    0.581395
bill_depth_mm             

### Antes dos exercícios: uma transformação por vez

Para entender uma operação vetorizada, faça primeiro `print(p['body_mass_g'].head(3))`: os valores são 3750, 3800 e 3250 gramas. Depois execute `print((p['body_mass_g'] / 1000).head(3))`: você verá 3.75, 3.80 e 3.25. A divisão foi aplicada a cada valor, sem um for. Crie então uma nova coluna: `q = p.assign(body_mass_kg=p['body_mass_g'] / 1000)`. O nome antes do sinal de igual dentro de assign define a nova coluna; a expressão depois define seus valores. q contém a tabela enriquecida e p preserva suas colunas originais.

Para texto, pense em uma sequência de etapas: Series → str → operação. Em `s.str.strip().str.lower()`, strip devolve os textos sem espaços externos; lower transforma o resultado em minúsculas. O encadeamento é apenas uma maneira compacta de escrever dois passos que você pode executar separadamente. Uma lista de valores entre colchetes vira uma Series com `pd.Series(...)`.

No exercício de tradução, map recebe um dicionário como `{'male': 'masculino', 'female': 'feminino'}`. Para cada valor da coluna, procura a chave e devolve a tradução correspondente. Antes de confiar no resultado, use `p['sex'].value_counts(dropna=False)` para conhecer as entradas. A chamada value_counts conta ocorrências de cada valor; dropna=False inclui as ausências no relatório.

### Aprofundamento e cuidados

Operações vetorizadas trabalham com uma coluna inteira: dividir a massa em gramas por mil cria quilogramas. `assign` devolve uma nova tabela, útil para preservar a anterior. Em uma sequência de assign, as expressões com lambda recebem a tabela daquele passo. Aqui lambda é uma função curta.

O acessor `str` oferece operações de texto: `strip` remove espaços nas extremidades, `lower` padroniza caixa e `contains` encontra padrões. Para buscas literais use `regex=False`; para valores ausentes escolha explicitamente `na=False` quando a regra for tratá-los como não correspondentes. `map` aplica um dicionário; chaves não encontradas viram ausências. Audite-as.

Evite `apply(axis=1)` quando soma, divisão, map, operações de texto ou condições vetorizadas resolvem. Antes de otimizar velocidade, confirme se unidades e regra estão corretas. Não some gramas com quilogramas e não calcule razões com denominador zero sem uma regra explícita.


## Exemplo guiado

In [3]:
enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

   body_mass_g  body_mass_kg
0       3750.0          3.75
1       3800.0          3.80
2       3250.0          3.25
3          NaN           NaN
4       3450.0          3.45
0    adelie
1    adelie
2    adelie
3    adelie
4    adelie
Name: species, dtype: string


## Exercício 04.1
Crie bill_length_cm a partir de bill_length_mm, preservando p.

In [4]:
q = p.assign(bill_length_cm=p['bill_length_mm']/10)
print(q[['bill_length_mm','bill_length_cm']].head())

   bill_length_mm  bill_length_cm
0            39.1            3.91
1            39.5            3.95
2            40.3            4.03
3             NaN             NaN
4            36.7            3.67


## Exercício 04.2
Padronize a Series didática [" Dream ", "BISCOE", None] em minúsculas e sem espaços externos.

In [5]:
s = pd.Series([' Dream ','BISCOE',None], dtype='string')
print(s.str.strip().str.lower())

0     dream
1    biscoe
2      <NA>
dtype: string


## Exercício 04.3
Traduza male/female para masculino/feminino em uma nova coluna. Preserve ausências.

In [6]:
q = p.assign(sexo_pt=p['sex'].map({'male':'masculino','female':'feminino'}))
print(q[['sex','sexo_pt']].head())

      sex    sexo_pt
0    male  masculino
1  female   feminino
2  female   feminino
3     NaN        NaN
4  female   feminino


## Exercício 04.4
Selecione espécies cujo nome contém a sequência literal elie, sem diferenciar caixa.

In [7]:
print(p.loc[p['species'].str.contains('elie', case=False, regex=False, na=False)])

    species     island  bill_length_mm  ...  body_mass_g     sex  year
0    Adelie  Torgersen            39.1  ...       3750.0    male  2007
1    Adelie  Torgersen            39.5  ...       3800.0  female  2007
2    Adelie  Torgersen            40.3  ...       3250.0  female  2007
3    Adelie  Torgersen             NaN  ...          NaN     NaN  2007
4    Adelie  Torgersen            36.7  ...       3450.0  female  2007
..      ...        ...             ...  ...          ...     ...   ...
147  Adelie      Dream            36.6  ...       3475.0  female  2009
148  Adelie      Dream            36.0  ...       3450.0  female  2009
149  Adelie      Dream            37.8  ...       3750.0    male  2009
150  Adelie      Dream            36.0  ...       3700.0  female  2009
151  Adelie      Dream            41.5  ...       4000.0    male  2009

[152 rows x 8 columns]


## Exercício 04.5
Crie a razão entre comprimento e profundidade do bico, protegendo denominadores iguais a zero.

In [8]:
den = p['bill_depth_mm'].where(p['bill_depth_mm'].ne(0))
print(p.assign(razao_bico=p['bill_length_mm']/den).head())

  species     island  bill_length_mm  ...     sex  year  razao_bico
0  Adelie  Torgersen            39.1  ...    male  2007    2.090909
1  Adelie  Torgersen            39.5  ...  female  2007    2.270115
2  Adelie  Torgersen            40.3  ...  female  2007    2.238889
3  Adelie  Torgersen             NaN  ...     NaN  2007         NaN
4  Adelie  Torgersen            36.7  ...  female  2007    1.901554

[5 rows x 9 columns]
